# E- handelsanalyysis

## Project syfte

Analysera försäljningsdata for att idetentifera använbara mönster och ge ledgning beslut innan nästa kampanjperiod.

## Frågeställningar

1. Vilka produkt kategorier driver mest intäkt?
2. Vilka städer står för störst intäkt?
3. Hur ser ut Försaljningen över tid?
4. Hur ser en typisk order ut? – AOV (Average Order Value) och spridning.
5. Topp-listor – topp-3 kategorier efter intäkt.
6. Finns det avvikelser eller något oväntat mönster i data?



In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys



In [17]:

import sys
from pathlib import Path

# Import the function
from src.io_utils import load_data

# Load the dataset
df = load_data("data/ecommerce_sales.csv")

print("Data loaded successfully!")


TypeError: load_data() takes 0 positional arguments but 1 was given

In [ ]:
df.head()

In [ ]:
print("Ecommerce data Information")
df.info()


In [ ]:
print("Statistics")

print(df.describe())
print()

In [ ]:

df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Date:" "From", df["date"] . min() , "to" ,df["date"].max())

In [ ]:
print("Cities:" ,list(df["city"].unique()))
print()

print("Produkt Categories:" ,list(df["category"].unique()))
print()

## Beräkna total Intäkten

In [ ]:
#Total intäkt och Antal enheter

total_intäkt = df["revenue"].sum()
print(f"Total intäkt: {total_intäkt:,.2f} kr")
print()

antal_enheter = df["units"].sum()
print(f"Total antal enheter: {antal_enheter}")


## Intäkt per stad

In [ ]:
#Intäkt Per Stad

#Tabell
import importlib
import src.metrics as metrics

stads_intäkt = metrics.intäkt_per_stad(df)

stads_intäkt_tabell = stads_intäkt.reset_index()
stads_intäkt_tabell.columns = ["Stad" , "Total intäkt"]

display(stads_intäkt_tabell)



In [ ]:
högst_intäkt_stad = stads_intäkt.idxmax()
hogsta_stads_intakt = stads_intäkt.max()

print(f"Stad med högst intäkt: {högst_intäkt_stad}")
print(f"Högsta stads intäkten: {hogsta_stads_intakt:,.2f} kr")

In [ ]:
# kategori intäkt 

In [ ]:
# Top tre kategori

In [ ]:
# månads intäkt

In [ ]:
#aov

In [ ]:
#ordervalue distribution


import importlib
import src.metrics as metrics

importlib.reload(metrics)

order_values = metrics.ordervarden(df)

print("Antal unika orderr:", order_values.size)
print(f"Lägsta ordervärde: {order_values.min():,.2f} kr")
print(f"Median för ordervärde: {order_values.median():,.2f} kr")
print(f"Högsta ordervärde: {order_values.max():,.2f} kr")

In [ ]:
nyckeltal_tabell = pd.DataFrame({
    "Nyckeltal": [
        "Total intäkt",
        "Totalt antal sålda enheter",
        "Antal unika ordrar",
        "Stad med högst intäkt",
    ],
    "Resultat": [
        f"{total_intäkt:,.2f} kr",
        f"{antal_enheter:,}",
        f"{order_values.size:,}",
        str(stads_intäkt.idxmax()),
        
    ]
})

display(nyckeltal_tabell)


# Visualization

In [ ]:
#plot intäkt per stad

import importlib
import src.viz as viz

importlib.reload(viz)

viz.plot_intakt_per_stad(stads_intäkt)

In [ ]:
#plot kategory

In [ ]:
#plot topp tre kategori

In [ ]:
#plot försäjlning over tid

In [ ]:
#Aov

In [ ]:
#order value distributin


import importlib
import src.viz as viz

importlib.reload(viz)

viz.plot_ordervarden(order_values)

## Analysera  avvikelser och något oväntat mönster


In [ ]:
# Saknande värden per kolumn
print(df.isnull().sum())

# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Invalid or unusual values
print("Price <= 0:", (df["price"] <= 0).sum())
print("Units <= 0:", (df["units"] <= 0).sum())
print("Revenue < 0:", (df["revenue"] < 0).sum())

In [ ]:

import numpy as np
import pandas as pd

kolumner = ["price", "units", "revenue"]

sammanfattningar = []
outlier_limit = {}
outlier_data = {}

for kolumn in kolumner:
    values = df[kolumn].dropna().to_numpy()

    minimum = np.min(values)
    q1 = np.percentile(values, 25)
    median = np.median(values)
    q3 = np.percentile(values, 75)
    maximum = np.max(values)

    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    # Identifiera potentiella avvikande värden
    mask = (df[kolumn] < lower_limit) | (
        df[kolumn] > upper_limit
    )
    avvikelser = df.loc[mask]

    sammanfattningar.append({
        "Kolumn": kolumn,
        "Minimum": minimum,
        "Q1": q1,
        "Median": median,
        "Q3": q3,
        "Maximum": maximum,
        "IQR": iqr,
        "Antal avvikelser": len(avvikelser)
    })

    outlier_limit[kolumn] = (
        lower_limit, upper_limit
    )
    outlier_data[kolumn] = avvikelser

sammanfattning = pd.DataFrame(sammanfattningar)

display(sammanfattning.round(2))


In [ ]:
alla_avvikelser = pd.concat(
    [outlier_data[kolumn] for kolumn in kolumner]
).drop_duplicates()

print(f"Totalt antal unika rader med avvikelser: {len(alla_avvikelser)}")

display(
    alla_avvikelser[
        ["order_id", "date", "city",
         "category", "price", "units", "revenue"]
    ].sort_values("revenue", ascending=False)
)


Datasetet innehåller 2 500 rader och 7 kolumner. Vi hittade inga saknade värden eller dubblettrader. Det fanns inte heller några negativa värden i kolumnerna price, units eller revenue.

Vi använde femtals-sammanfattningen och IQR-metoden för att identifiera potentiellt avvikande värden. Analysen visade 88 avvikande värden i price, 7 i units och 111 i revenue.

Resultaten visar att kategorin Electronics har flest avvikande värden, både för pris och intäkt. Detta kan bero på att elektronikprodukter ofta har högre priser och att försäljning av flera enheter kan ge högre intäkter.